# 7주차 워크북 1
제공 PDF는 Google Drive의 [전체 ZIP](https://drive.google.com/file/d/1SYb8OH0cPCH8kjgq3arDDYvoWm_gB85f/view)에서 `Week07/data/week07_course_guide.pdf`를 받아 Colab에 업로드하세요.


In [ ]:
!pip -q install pypdf scikit-learn


In [ ]:
from pathlib import Path
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
file = Path("week07_course_guide.pdf")
assert file.is_file(), "먼저 PDF를 Colab에 업로드하세요."
reader = PdfReader(str(file))
records = [(p, (page.extract_text() or "").strip())
           for p, page in enumerate(reader.pages, start=1)]
records = [(p, t) for p, t in records if t]
assert records, "스캔 PDF에서는 별도의 OCR이 필요합니다."
print("추출 페이지 수:", len(records))



In [ ]:
def chunk_words(text, size=75, overlap=20):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("0 <= overlap < size 조건이 필요합니다.")
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size])
            for i in range(0, len(words), step)
            if words[i:i + size]]

chunks = [{"page": p, "text": part}
          for p, text in records
          for part in chunk_words(text)]
print("청크 수:", len(chunks))



In [ ]:
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
vectors = vectorizer.fit_transform([x["text"] for x in chunks])

def retrieve(question, top_k=3):
    query = vectorizer.transform([question])
    scores = cosine_similarity(query, vectors).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[int(i)], "score": float(scores[int(i)])}
            for i in order]

for question in ["RAG가 무엇인가요?", "API 키는 어떻게 관리하나요?",
                 "청크를 왜 만드나요?"]:
    best = retrieve(question, 1)[0]
    print(question, "페이지:", best["page"],
          "유사도:", round(best["score"], 3))



## 학생 과제
질문 세 개의 근거 페이지와 점수를 기록합니다. 청크 크기 변경 후 비교하고, 워크북 2는 UI 화면을 캡처하세요.
핵심 코드: PdfReader, extract_text, TfidfVectorizer, cosine_similarity, gr.Interface.
